# 04 – Feature Engineering

Cyclical day-of-year encoding from `date_of_record`.

- Adds `doy_sin`, `doy_cos`
- Drops redundant `month` / `season` text columns
- Keeps `station_name`, `state`, `district` as metadata (not model inputs yet)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths whether the notebook is run from notebooks/ or project root
BASE = Path("..").resolve()
if not (BASE / "data" / "processed" / "clean_dataset.csv").exists():
    BASE = Path(".").resolve()

CLEAN_PATH = BASE / "data" / "processed" / "clean_dataset.csv"
OUT_PATH = BASE / "data" / "processed" / "feature_engineered.csv"

# 1. Load cleaned dataset
df = pd.read_csv(CLEAN_PATH)
df["date_of_record"] = pd.to_datetime(df["date_of_record"])

print("Loaded:", df.shape)
df.head()

In [ ]:
# 2. Sanity check before we touch anything
n_rows_before = len(df)
assert df["date_of_record"].isna().sum() == 0, "Found unparseable dates — stop and investigate"
print(f"Rows: {n_rows_before:,}")
print("Columns:", df.columns.tolist())

In [ ]:
# 3. Day-of-year cyclical encoding
# use 366 so leap years don't distort the phase
day_of_year = df["date_of_record"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * day_of_year / 366)
df["doy_cos"] = np.cos(2 * np.pi * day_of_year / 366)

# 4. Drop redundant categorical calendar columns
df = df.drop(columns=["month", "season"])

# 5. state / district / station_name kept as metadata (not encoded here)

In [ ]:
# 6. Verification before saving
assert len(df) == n_rows_before, "Row count changed - something went wrong"
assert df["doy_sin"].between(-1, 1).all(), "doy_sin out of expected range"
assert df["doy_cos"].between(-1, 1).all(), "doy_cos out of expected range"
assert df.isna().sum().sum() == 0, "NaNs introduced during feature engineering"

# 7. Save
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df):,} rows, {df.shape[1]} columns -> {OUT_PATH}")
print(df[["date_of_record", "doy_sin", "doy_cos"]].head())

## Post-save verification

In [ ]:
check = pd.read_csv(OUT_PATH)
print(check.shape)
print(check.columns.tolist())
print(check[["doy_sin", "doy_cos"]].describe())